In [ ]:
import pandas as pd

# Load the CSV file into a DataFrame
try:
    df = pd.read_csv('data1.csv')
    print("File loaded successfully.")
    # Display the first 5 rows of the DataFrame
    display(df.head())
except FileNotFoundError:
    print("Error: 'data1.csv' not found. Please make sure the file is in the '/content/' directory.")
except Exception as e:
    print(f"An error occurred while loading the file: {e}")

Now that the data is loaded, we need to understand its structure before developing an ANFIS model. This typically involves:

1.  **Exploring data types and missing values**: To identify columns that might need cleaning or conversion.
2.  **Identifying input and output variables**: Determining which columns will be used as features (inputs) and which as the target (output) for the ANFIS model.
3.  **Data preprocessing**: This could include normalization, scaling, or feature engineering depending on the data characteristics and ANFIS requirements.

Let's start by looking at the basic information and descriptive statistics of the DataFrame.

In [ ]:
# Display general information about the DataFrame, including data types and non-null values
df.info()

# Display descriptive statistics of the numerical columns
display(df.describe())

In [ ]:
from sklearn.model_selection import train_test_split

# Define target variable (y) and features (X)
y = df['Maioria']
X = df.drop('Maioria', axis=1)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

# Display the first few rows of the training features and target
print("\nFirst 5 rows of X_train:")
display(X_train.head())
print("\nFirst 5 rows of y_train:")
display(y_train.head())

The data is now split into training and testing sets. The next crucial step for developing an ANFIS model is to **define the ANFIS architecture and parameters**. This involves:

1.  **Determining the number and type of membership functions** for each input variable.
2.  **Specifying the fuzzy inference system structure** (e.g., Sugeno or Mamdani).
3.  **Choosing an optimization algorithm** for training.

In [ ]:
from sklearn.metrics import cohen_kappa_score

print("\n--- Individual Metric Evaluation ---")

# Evaluate each feature as a predictor for 'Maioria'
for col in X_test.columns:
    # Use the test data for evaluation
    y_true = y_test

    # For binary features (int64) or features that are already 0/1 like 'EX', 'EM', etc.
    # or for continuous features, we apply a threshold of 0.5 to convert to binary prediction.
    if df[col].dtype == 'int64' or df[col].dtype == 'float64':
        y_pred_col = (X_test[col] >= 0.5).astype(int)
    else:
        # For other data types, this approach might not be suitable
        print(f"Skipping evaluation for '{col}' due to unsupported data type: {df[col].dtype}")
        continue

    accuracy_col = accuracy_score(y_true, y_pred_col)
    kappa_col = cohen_kappa_score(y_true, y_pred_col)

    print(f"\nMetric: {col}")
    print(f"  Accuracy: {accuracy_col:.3f}")
    print(f"  Cohen's Kappa: {kappa_col:.3f}")

## Construindo um ANFIS Customizado com PyTorch

Dado que as bibliotecas ANFIS dedicadas não foram instaladas com sucesso, podemos abordar a parte 'neuro-adaptativa' do ANFIS construindo uma arquitetura customizada usando um framework de deep learning como o PyTorch. Isso nos permite tratar as funções de pertinência e as regras fuzzy como camadas parametrizadas em uma rede neural, que podem ser otimizadas usando gradiente descendente.

### Estrutura do ANFIS como uma Rede Neural

Um ANFIS pode ser conceitualmente dividido em 5 camadas:

1.  **Camada de Fuzzificação**: Cada entrada numérica é convertida em graus de pertinência a diferentes conjuntos fuzzy (e.g., 'baixo', 'médio', 'alto'). As funções de pertinência (e.g., Gaussianas, triangulares) têm parâmetros (médias, desvios padrão, pontos de pico) que serão aprendidos.
2.  **Camada de Regras**: Nesta camada, os graus de pertinência são combinados para determinar a força de ativação de cada regra fuzzy. Geralmente, usa-se o produto (AND fuzzy) ou o mínimo (AND fuzzy).
3.  **Camada de Normalização**: As forças de ativação das regras são normalizadas para que a soma de todas as forças de ativação seja 1.
4.  **Camada de Consequentes**: Cada regra fuzzy tem um consequente. Para um modelo Sugeno (comumente usado em ANFIS), o consequente é um polinômio (geralmente linear ou constante) das entradas. Os coeficientes desses polinômios também serão aprendidos.
5.  **Camada de Defuzzificação**: As saídas de cada regra (consequentes ponderados pela força de ativação normalizada) são combinadas para produzir uma única saída numérica final.

In [ ]:
import sys
import struct

print(f"Versão do Python: {sys.version}")
print(f"Arquitetura: {struct.calcsize('P') * 8} bits")

%pip install --upgrade pip

In [ ]:
# Instalar PyTorch se não estiver presente (geralmente já está no Colab)
try:
    import torch
    print("PyTorch já está instalado.")
except ImportError:
    print("Instalando PyTorch...")
    %pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
    import torch
    print("PyTorch instalado com sucesso.")


### Implementação de uma Camada ANFIS Básica (Exemplo com Funções de Pertinência Gaussianas)

Vamos construir um modelo ANFIS simplificado para classificar a 'Maioria' com base nas nossas três features de entrada (`Soft_F1`, `Stats`, `Similarity`).

Neste exemplo, usaremos:
- Funções de pertinência Gaussianas para a camada de fuzzificação.
- Multiplicação para a operação AND na camada de regras.
- Um consequente constante para cada regra (simplificando a parte Sugeno para um valor direto a ser aprendido).

Este é um modelo muito básico e pode ser expandido para consequentes lineares e diferentes tipos de funções de pertinência.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class GaussianMembershipFunction(nn.Module):
    def __init__(self, num_mfs):
        super(GaussianMembershipFunction, self).__init__()
        self.means = nn.Parameter(torch.linspace(0, 1, num_mfs).repeat(1, 1))
        self.stds = nn.Parameter(torch.ones(1, num_mfs) * 0.1)

    def forward(self, x):
        x_expanded = x.expand(-1, self.means.shape[1])
        return torch.exp(-torch.pow(x_expanded - self.means, 2) / (2 * torch.pow(self.stds + 1e-6, 2)))


class ANFIS(nn.Module):
    def __init__(self, num_inputs, num_mfs_per_input, num_outputs=1):
        super(ANFIS, self).__init__()
        self.num_inputs = num_inputs
        self.num_mfs_per_input = num_mfs_per_input
        self.num_rules = num_mfs_per_input ** num_inputs
        self.num_outputs = num_outputs

        # Layer 1: Fuzzification
        self.fuzz_layers = nn.ModuleList([
            GaussianMembershipFunction(num_mfs_per_input) for _ in range(num_inputs)
        ])

        # Layer 4: Consequents (Modified: Linear function for each rule)
        # For each rule, we need (num_inputs + 1) parameters (coefficients for each input + a bias term)
        self.consequents = nn.Parameter(torch.randn(self.num_rules, num_inputs + 1))

    def forward(self, x):
        # x: (batch_size, num_inputs)

        # Layer 1: Fuzzification
        mf_outputs = []
        for i in range(self.num_inputs):
            mf_outputs.append(self.fuzz_layers[i](x[:, i].unsqueeze(1)))

        # Layer 2: Rule Layer (T-norm: product operator)
        rule_activations = []
        for i in range(self.num_rules):
            rule_activation = torch.ones(x.shape[0], 1).to(x.device)
            for j in range(self.num_inputs):
                mf_idx = (i // (self.num_mfs_per_input ** (self.num_inputs - 1 - j))) % self.num_mfs_per_input
                rule_activation = rule_activation * mf_outputs[j][:, mf_idx].unsqueeze(1)
            rule_activations.append(rule_activation)

        rule_activations = torch.cat(rule_activations, dim=1)

        # Layer 3: Normalization Layer
        sum_rule_activations = torch.sum(rule_activations, dim=1, keepdim=True)
        normalized_activations = rule_activations / (sum_rule_activations + 1e-6)

        # Layer 5: Defuzzification (Weighted average of linear consequents)
        # Calculate consequent output for each rule: f_k = a_k * x1 + b_k * x2 + ... + c_k
        # For each rule, the consequent is a linear combination of inputs and a bias.
        # x_with_bias: (batch_size, num_inputs + 1) - add a column of ones for the bias term
        x_with_bias = torch.cat([x, torch.ones(x.shape[0], 1).to(x.device)], dim=1)

        # consequent_outputs: (batch_size, num_rules)
        consequent_outputs_per_rule = torch.matmul(x_with_bias, self.consequents.transpose(0, 1))

        # Final output is the sum of (normalized_activation * consequent_output_per_rule)
        output = torch.sum(normalized_activations * consequent_outputs_per_rule, dim=1, keepdim=True)

        return output

### Preparação dos Dados e Treinamento

Agora vamos preparar nossos dados (`X_train`, `y_train`, `X_test`, `y_test`) para PyTorch, instanciar o modelo ANFIS e definir um loop de treinamento. Usaremos uma função de perda `MSELoss` para simplificar, mas para classificação binária, `BCEWithLogitsLoss` seria mais apropriado, seguida de uma função de ativação sigmoide na saída do modelo.

Para a 'Maioria' (0 ou 1), podemos tratar isso como um problema de regressão simples para demonstrar, onde a saída do ANFIS deve ser próxima de 0 ou 1, e depois aplicar um limiar (e.g., 0.5) para obter a classificação binária.


In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Remover 'VES', 'StCo' e 'LFA' da lista de features
features_to_remove = ['VES', 'StCo', 'LFA']
input_features_anfis = [f for f in X_train.columns.tolist() if f not in features_to_remove]

# Converter dados para tensores PyTorch com o novo conjunto de features
X_train_tensor = torch.tensor(X_train[input_features_anfis].values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test[input_features_anfis].values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

# Instanciar o modelo ANFIS com o novo num_inputs
num_inputs = len(input_features_anfis)
num_mfs_per_input = 2
anfis_model = ANFIS(num_inputs=num_inputs, num_mfs_per_input=num_mfs_per_input, num_outputs=1)

# Definir função de perda e otimizador
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(anfis_model.parameters(), lr=0.01)

# Loop de treinamento
num_epochs = 200

print(f"Iniciando treinamento do ANFIS (sem {features_to_remove}) - Total de features: {num_inputs}")
for epoch in range(num_epochs):
    anfis_model.train()
    optimizer.zero_grad()

    outputs = anfis_model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)

    loss.backward()
    optimizer.step()

    if (epoch+1) % 20 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

print("Treinamento do ANFIS concluído.")

### Avaliação do ANFIS Treinado

Após o treinamento, vamos avaliar o desempenho do nosso modelo ANFIS customizado no conjunto de teste. Para a classificação binária, aplicaremos um limiar de 0.5 à saída contínua do modelo.

In [ ]:
from sklearn.metrics import accuracy_score, cohen_kappa_score, classification_report

anfis_model.eval() # Colocar o modelo em modo de avaliação
with torch.no_grad(): # Desativar cálculo de gradientes
    test_outputs = anfis_model(X_test_tensor)
    # Converter saídas contínuas para previsões binárias
    anfis_predictions = (torch.sigmoid(test_outputs) >= 0.5).int().squeeze().numpy() # Apply sigmoid as BCEWithLogitsLoss implies raw logits

# Avaliar o desempenho
accuracy = accuracy_score(y_test_tensor.squeeze().numpy(), anfis_predictions)
kappa = cohen_kappa_score(y_test_tensor.squeeze().numpy(), anfis_predictions)

print("\n--- Avaliação do ANFIS Customizado (PyTorch) no Conjunto de Teste ---")
print(f"Acurácia: {accuracy:.3f}")
print(f"Cohen's Kappa: {kappa:.3f}")
print("\nRelatório de Classificação:")
print(classification_report(y_test_tensor.squeeze().numpy(), anfis_predictions))

In [ ]:
import pandas as pd

# Create a tensor from the full feature set X
X_full_tensor = torch.tensor(X[input_features_anfis].values, dtype=torch.float32)

# Put the model in evaluation mode
anfis_model.eval()

# Get predictions for the entire dataset
with torch.no_grad():
    full_outputs = anfis_model(X_full_tensor)
    # Convert continuous outputs to binary predictions
    anfis_full_predictions = (torch.sigmoid(full_outputs) >= 0.5).int().squeeze().numpy()

# Add the predictions to the original DataFrame for display
df_with_anfis_predictions = df.copy()
df_with_anfis_predictions['ANFIS_Predicted_Maioria'] = anfis_full_predictions

print("Original DataFrame with ANFIS predictions for the full dataset:")
display(df_with_anfis_predictions.head())

In [ ]:
import pandas as pd

# Create a tensor from the full feature set X
X_full_tensor = torch.tensor(X[input_features_anfis].values, dtype=torch.float32)

# Put the model in evaluation mode
anfis_model.eval()

# Get predictions for the entire dataset
with torch.no_grad():
    full_outputs = anfis_model(X_full_tensor)
    # Get continuous outputs (after sigmoid activation)
    anfis_full_continuous_predictions = torch.sigmoid(full_outputs).squeeze().numpy()

# Add the continuous predictions to the original DataFrame for display
df_with_anfis_continuous_predictions = df.copy()
df_with_anfis_continuous_predictions['ANFIS_Continuous_Maioria'] = anfis_full_continuous_predictions

print("Original DataFrame with ANFIS continuous predictions for the full dataset:")
display(df_with_anfis_continuous_predictions.head())

In [ ]:
# Save the DataFrame with ANFIS continuous predictions to a CSV file
df_with_anfis_continuous_predictions.to_csv('data3.csv', index=False)

print("DataFrame with ANFIS full continuous predictions for data1.csv saved to 'data3.csv'")

Now, let's load `data2.csv` and generate continuous ANFIS predictions for it.

In [ ]:
import pandas as pd

# Load the CSV file into a DataFrame
try:
    df_data2 = pd.read_csv('data2.csv')
    print("File 'data2.csv' loaded successfully.")
    # Display the first 5 rows of the DataFrame
    display(df_data2.head())
except FileNotFoundError:
    print("Error: 'data2.csv' not found. Please make sure the file is in the '/content/' directory.")
except Exception as e:
    print(f"An error occurred while loading the file: {e}")

In [ ]:
# Prepare the features from df_data2 for ANFIS prediction
# Ensure the columns match the training features (input_features_anfis)

# Drop 'Maioria' if it exists in data2.csv, as it's the target
if 'Maioria' in df_data2.columns:
    X_data2 = df_data2.drop('Maioria', axis=1)
else:
    X_data2 = df_data2.copy()

# Ensure X_data2 has the same columns and order as input_features_anfis
# This is crucial for consistent predictions
X_data2_tensor = torch.tensor(X_data2[input_features_anfis].values, dtype=torch.float32)

print(f"Features from 'data2.csv' prepared with shape: {X_data2_tensor.shape}")

In [ ]:
# Generate continuous ANFIS predictions for data2.csv
anfis_model.eval() # Ensure model is in evaluation mode
with torch.no_grad():
    data2_outputs = anfis_model(X_data2_tensor)
    anfis_data2_continuous_predictions = torch.sigmoid(data2_outputs).squeeze().numpy()

# Add these continuous predictions to df_data2
df_data2_with_anfis_predictions = df_data2.copy()
df_data2_with_anfis_predictions['ANFIS_Continuous_Maioria'] = anfis_data2_continuous_predictions

print("DataFrame 'data2.csv' with ANFIS continuous predictions:")
display(df_data2_with_anfis_predictions.head())

In [ ]:
# Save the DataFrame with ANFIS continuous predictions for data2.csv to a new CSV file
df_data2_with_anfis_predictions.to_csv('data4.csv', index=False)

print("DataFrame with ANFIS continuous predictions for data2.csv saved to 'data4.csv'")

In [ ]:
# Save the DataFrame with ANFIS predictions to a CSV file
df_with_anfis_predictions.to_csv('data5.csv', index=False)

print("DataFrame with ANFIS full predictions saved to 'data5.csv'")

In [ ]:
import pandas as pd

# Create a DataFrame for fuzzy system predictions
fuzzy_results_df = X_test[input_features_anfis].copy() # Copy input features used by ANFIS
fuzzy_results_df['True_Maioria'] = y_test.values
fuzzy_results_df['Fuzzy_Predicted_Maioria'] = anfis_predictions

# Save the DataFrame to a CSV file
fuzzy_results_df.to_csv('fuzzy_predictions_results.csv', index=False)

print("Fuzzy system prediction results saved to 'fuzzy_predictions_results.csv'")
display(fuzzy_results_df.head())

Este exemplo mostra a base de como você poderia implementar um ANFIS customizado usando PyTorch. É importante notar que este é um modelo bastante simplificado. Um ANFIS completo incluiria:

*   **Consequentes Polinomiais**: Em vez de apenas uma constante, os consequentes seriam funções lineares das entradas.
*   **Mais Funções de Pertinência**: O número e tipo de MFs poderiam ser ajustados.
*   **Regularização**: Para evitar overfitting.
*   **Validação Cruzada**: Para uma avaliação mais robusta do modelo.

Esta abordagem oferece flexibilidade, mas também exige um conhecimento mais aprofundado de PyTorch e dos princípios do ANFIS.

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    # O comando abaixo diz ao LaTeX para usar pacotes com fontes estilo Times
    "text.latex.preamble": r"\usepackage{mathptmx}",
    "axes.labelsize": 16,
    "font.size": 14,
    "legend.fontsize": 14,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
})

### Análise dos Parâmetros Aprendidos do ANFIS

Vamos inspecionar os parâmetros do modelo ANFIS que foram otimizados durante o treinamento. Isso nos dará uma visão das funções de pertinência aprendidas e dos consequentes das regras.

In [ ]:
# Instalar scikit-fuzzy se não estiver presente
%pip install scikit-fuzzy

import matplotlib.pyplot as plt
import numpy as np
import skfuzzy as fuzz # Importar a biblioteca scikit-fuzzy

# Exibir os parâmetros das funções de pertinência Gaussianas
print("\n--- Parâmetros das Funções de Pertinência Gaussianas ---")
for i, layer in enumerate(anfis_model.fuzz_layers):
    feature_name = input_features_anfis[i]
    print(f"Feature: {feature_name}")
    print(f"  Means: {layer.means.detach().cpu().numpy().flatten()}")
    print(f"  Stds: {layer.stds.detach().cpu().numpy().flatten()}")

    # Plotar as funções de pertinência aprendidas
    x_universe = np.arange(0, 1.01, 0.01)
    plt.figure(figsize=(8, 4))
    for j in range(anfis_model.num_mfs_per_input):
        mean = layer.means.detach().cpu().numpy().flatten()[j]
        std = layer.stds.detach().cpu().numpy().flatten()[j]
        y_mf = fuzz.gaussmf(x_universe, mean, std)
        plt.plot(x_universe, y_mf, label=f'MF {j+1} (Mean={mean:.2f}, Std={std:.2f})')
    plt.title(f'Funções de Pertinência Aprendidas para {feature_name}')
    plt.xlabel('Valor de Entrada')
    plt.ylabel('Grau de Pertinência')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)
# --- NOVO: Salvar como PDF dinamicamente para cada feature ---
    nome_arquivo = f"results/Funcoes_Pertinencia_{feature_name}.pdf"
    plt.savefig(nome_arquivo, format='pdf', bbox_inches='tight')

    plt.show()


# Exibir os parâmetros dos consequentes das regras
print("\n--- Parâmetros dos Consequentes das Regras (Coeficientes Lineares) ---")
print("Cada linha corresponde a uma regra. As colunas são os coeficientes para cada entrada e o termo de bias.")
consequents_numpy = anfis_model.consequents.detach().cpu().numpy()
consequent_columns = input_features_anfis + ['Bias']
consequent_df = pd.DataFrame(consequents_numpy, columns=consequent_columns)
display(consequent_df.head())

print("\n--- Interpretação da Importância das Regras ---")
print("Os pesos (coeficientes) dos consequentes indicam a contribuição de cada regra para a saída final. Regras com maiores valores absolutos em seus coeficientes podem ser consideradas mais 'importantes' para a decisão final, especialmente se forem ativadas frequentemente por dados de entrada.")
print("Para uma análise mais aprofundada da 'importância' das regras, seria necessário analisar a força de ativação média de cada regra para todo o dataset e multiplicar pela magnitude de seus consequentes. No entanto, os coeficientes por si só já dão uma boa indicação do comportamento de cada regra.")

In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import cohen_kappa_score

def calcular_importancia_permutacao_media(modelo, X_test, y_test, features_nomes, n_iteracoes=10):
    """
    Calcula a importância de cada feature embaralhando seus valores múltiplas vezes
    e medindo a queda média no Kappa de Cohen, com desvio padrão.
    """
    # Garante que os resultados sejam sempre reprodutíveis
    torch.manual_seed(43)

    # Coloca o modelo em modo de avaliação (desativa dropout/batchnorm e previne atualizações)
    modelo.eval()

    # 1. Baseline: Previsão original sem alterações (Modelo Saudável)
    with torch.no_grad():
        saida_base = modelo(X_test).squeeze().cpu().numpy()
        # Assume-se um limiar de 0.5 para classificação binária na saída contínua do ANFIS
        pred_base = (saida_base > 0.5).astype(int)
        y_true = y_test.cpu().numpy()

        # Calcula o Kappa original
        kappa_base = cohen_kappa_score(y_true, pred_base)

    print(f"Kappa Baseline (Original no Teste): {kappa_base:.4f}\n")

    resultados_importancia = []

    # 2. Permutação métrica a métrica
    for i, feature in enumerate(features_nomes):
        quedas_kappa = []

        # Repete o embaralhamento N vezes para calcular a média
        for _ in range(n_iteracoes):
            # Cria uma cópia limpa dos dados de teste a cada iteração
            X_temp = X_test.clone()

            # Embaralha APENAS a coluna 'i'
            indices_embaralhados = torch.randperm(X_temp.size(0))
            X_temp[:, i] = X_temp[indices_embaralhados, i]

            # 3. Nova previsão com a métrica "quebrada"
            with torch.no_grad():
                saida_perm = modelo(X_temp).squeeze().cpu().numpy()
                pred_perm = (saida_perm > 0.5).astype(int)

                # Calcula o novo Kappa
                kappa_perm = cohen_kappa_score(y_true, pred_perm)

            # Armazena a queda desta iteração específica
            quedas_kappa.append(kappa_base - kappa_perm)

        # Calcula a média e o desvio padrão das quedas para esta feature
        media_queda = np.mean(quedas_kappa)
        std_queda = np.std(quedas_kappa)

        resultados_importancia.append({
            'Métrica': feature,
            'Queda_Media_Kappa': media_queda,
            'Desvio_Padrao': std_queda
        })

    # 4. Formatação e visualização dos resultados
    df_imp = pd.DataFrame(resultados_importancia)
    # Ordena da métrica mais importante (maior queda média) para a menos importante
    df_imp = df_imp.sort_values(by='Queda_Media_Kappa', ascending=False)

    print(f"--- Importância Média das Métricas ({n_iteracoes} iterações) ---")
    display(df_imp)

    # Plotagem
    plt.figure(figsize=(7, 5))

    # Adicionamos yerr para as barras de erro e capsize para o tracinho na ponta
    plt.bar(df_imp['Métrica'], df_imp['Queda_Media_Kappa'],
            yerr=df_imp['Desvio_Padrao'], capsize=5, color='#1f77b4', alpha=0.85)

    plt.title(f'Importância Global das Métricas no ANFIS')
    plt.ylabel('Redução Média no Kappa de Cohen')
    plt.xticks(rotation=45)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout() # Garante que os rótulos não sejam cortados
    plt.show()

    return df_imp

# =====================================================================
# EXECUÇÃO NO NOTEBOOK
# =====================================================================
# Certifique-se de usar os tensores de teste/validação reais
# Você pode aumentar n_iteracoes (ex: 20 ou 30) se quiser ainda mais estabilidade

df_resultados = calcular_importancia_permutacao_media(
    modelo=anfis_model,
    X_test=X_test_tensor,
    y_test=y_test_tensor,
    features_nomes=input_features_anfis,
    n_iteracoes=20
)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import cohen_kappa_score

def fixar_sementes(semente):
    """Garante reprodutibilidade para uma semente específica."""
    random.seed(semente)
    np.random.seed(semente)
    torch.manual_seed(semente)
    torch.cuda.manual_seed(semente)
    torch.cuda.manual_seed_all(semente)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# =====================================================================
# 1. PREPARAÇÃO DOS DADOS
# =====================================================================
features_to_remove = ['VES', 'StCo', 'LFA']
input_features_anfis = [f for f in X_train.columns.tolist() if f not in features_to_remove]
num_inputs = len(input_features_anfis)

X_train_tensor = torch.tensor(X_train[input_features_anfis].values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test[input_features_anfis].values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)
y_true = y_test_tensor.cpu().numpy()

# =====================================================================
# 2. LOOP DE MÚLTIPLOS TREINAMENTOS E PERMUTAÇÕES
# =====================================================================
# Definimos 5 sementes diferentes para inicializar 5 modelos distintos
sementes_treinamento = [42, 123, 999, 2026, 7]
n_permutações_por_modelo = 10

# Dicionário para acumular as quedas de Kappa de TODAS as rodadas
todas_quedas = {feature: [] for feature in input_features_anfis}

print(f"--- Iniciando Experimento Robusto ({len(sementes_treinamento)} treinamentos independentes) ---")

for idx, semente in enumerate(sementes_treinamento):
    print(f"\n[{idx+1}/{len(sementes_treinamento)}] Treinando modelo com Semente {semente}...")
    fixar_sementes(semente)

    # 2.1 Instanciar modelo DO ZERO
    anfis_model = ANFIS(num_inputs=num_inputs, num_mfs_per_input=2, num_outputs=1)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(anfis_model.parameters(), lr=0.01)

    # 2.2 Treinar o modelo
    anfis_model.train()
    for epoch in range(200):
        optimizer.zero_grad()
        outputs = anfis_model(X_train_tensor)
        loss = criterion(outputs, y_train_tensor)
        loss.backward()
        optimizer.step()

    # 2.3 Calcular Baseline deste modelo específico
    anfis_model.eval()
    with torch.no_grad():
        saida_base = anfis_model(X_test_tensor).squeeze().cpu().numpy()
        pred_base = (saida_base > 0.5).astype(int)
        kappa_base = cohen_kappa_score(y_true, pred_base)

    print(f"      -> Kappa Baseline alcançado: {kappa_base:.4f}")

    # 2.4 Executar Importância por Permutação neste modelo
    for i, feature in enumerate(input_features_anfis):
        for _ in range(n_permutações_por_modelo):
            X_temp = X_test_tensor.clone()

            # Embaralha APENAS a coluna 'i'
            indices_embaralhados = torch.randperm(X_temp.size(0))
            X_temp[:, i] = X_temp[indices_embaralhados, i]

            with torch.no_grad():
                saida_perm = anfis_model(X_temp).squeeze().cpu().numpy()
                pred_perm = (saida_perm > 0.5).astype(int)
                kappa_perm = cohen_kappa_score(y_true, pred_perm)

            # Salva a queda globalmente
            todas_quedas[feature].append(kappa_base - kappa_perm)

# =====================================================================
# 3. AGREGAÇÃO ESTRATÍSTICA FINAL E VISUALIZAÇÃO
# =====================================================================
resultados_finais = []
for feature in input_features_anfis:
    quedas = todas_quedas[feature]
    resultados_finais.append({
        'Métrica': feature,
        'Queda_Media_Kappa': np.mean(quedas),
        'Desvio_Padrao': np.std(quedas)
    })

df_imp = pd.DataFrame(resultados_finais)
df_imp = df_imp.sort_values(by='Queda_Media_Kappa', ascending=False)

print("\n" + "="*50)
print(f"--- RESULTADO GLOBAL ({len(sementes_treinamento)} Modelos x {n_permutações_por_modelo} Permutações) ---")
print("="*50)
display(df_imp)

plt.figure(figsize=(8, 5))
plt.bar(df_imp['Métrica'], df_imp['Queda_Media_Kappa'],
        yerr=df_imp['Desvio_Padrao'], capsize=5, color='#1f77b4', alpha=0.85)

plt.title(f'Importância das Métricas no ANFIS (Variação de Treinamento)')
plt.ylabel('Redução Média no Kappa de Cohen')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
print("\n" + "="*50)
print(f"--- RESULTADO GLOBAL ({len(sementes_treinamento)} Modelos x {n_permutações_por_modelo} Permutações) ---")
print("="*50)
display(df_imp)

plt.figure(figsize=(7, 4))
plt.bar(df_imp['Métrica'], df_imp['Queda_Media_Kappa'],
        yerr=df_imp['Desvio_Padrao'], capsize=5, color='#1f77b4', alpha=0.85)

plt.title(f'Importância das Métricas no ANFIS')
plt.ylabel('Redução Média no Kappa')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()

# --- NOVO: Salvar como PDF ---
plt.savefig('results/Importancia_Metricas_ANFIS.pdf', format='pdf', bbox_inches='tight')

plt.show()

In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
torch.manual_seed(42)
# 1. Calcular a ativação (Firing Strength) de cada regra para todo o dataset
anfis_model.eval()
with torch.no_grad():
    # Precisamos capturar as ativações normalizadas.
    # Vamos replicar a lógica do forward para pegar 'normalized_activations'
    x = X_full_tensor
    mf_outputs = []
    for i in range(anfis_model.num_inputs):
        mf_outputs.append(anfis_model.fuzz_layers[i](x[:, i].unsqueeze(1)))

    rule_activations = []
    for i in range(anfis_model.num_rules):
        rule_activation = torch.ones(x.shape[0], 1)
        for j in range(anfis_model.num_inputs):
            mf_idx = (i // (anfis_model.num_mfs_per_input ** (anfis_model.num_inputs - 1 - j))) % anfis_model.num_mfs_per_input
            rule_activation = rule_activation * mf_outputs[j][:, mf_idx].unsqueeze(1)
        rule_activations.append(rule_activation)

    rule_activations = torch.cat(rule_activations, dim=1)
    sum_rule_activations = torch.sum(rule_activations, dim=1, keepdim=True)
    normalized_activations = rule_activations / (sum_rule_activations + 1e-6)

# 2. Calcular o Impacto Real: Ativação Média * Magnitude do Consequente
ativacao_media = torch.mean(normalized_activations, dim=0).cpu().numpy()
magnitude_consequente = np.abs(anfis_model.consequents.detach().cpu().numpy()[:, :-1]).mean(axis=1)
impacto_real_regra = ativacao_media * magnitude_consequente

# 3. Criar DataFrame de Regras
regras_indices = np.arange(anfis_model.num_rules)
ranking_regras = pd.DataFrame({
    'ID_Regra': regras_indices,
    'Ativacao_Media': ativacao_media,
    'Impacto_no_Modelo': impacto_real_regra
}).sort_values(by='Impacto_no_Modelo', ascending=False)

print("--- Top 10 Regras Mais Influentes (Frequência x Magnitude) ---")
display(ranking_regras.head(10))

# 4. Visualização do Top 20
plt.figure(figsize=(7, 3))
plt.bar(range(20), ranking_regras['Impacto_no_Modelo'].iloc[:20], color='salmon')
plt.xticks(range(20), ranking_regras['ID_Regra'].iloc[:20], rotation=90)
plt.title('As 20 Regras com Maior Impacto Real no Dataset')
plt.xlabel('ID da Regra (Combinação Fuzzy)')
plt.ylabel('Ativação x Coeficiente')
plt.grid(axis='y', alpha=0.3)
plt.savefig('results/Top20_Impacto_Regras.pdf', format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
def decodificar_regra(rule_id, num_inputs, num_mfs_per_input, feature_names):
    condicoes = []
    for j in range(num_inputs):
        # Lógica para encontrar qual MF cada entrada usa nesta regra específica
        mf_idx = (rule_id // (num_mfs_per_input ** (num_inputs - 1 - j))) % num_mfs_per_input
        # No nosso modelo, índice 0 é 'Baixo' e 1 é 'Alto'
        mf_nome = "Baixo" if mf_idx == 0 else "Alto"
        condicoes.append(f"{feature_names[j]} is {mf_nome}")
    return " IF " + " AND ".join(condicoes)

# Pegar o Top 5 do ranking de impacto real
top_regras = ranking_regras.head(5)

print("--- DECODIFICAÇÃO DAS REGRAS MAIS INFLUENTES ---")
for idx, row in top_regras.iterrows():
    r_id = int(row['ID_Regra'])
    logica = decodificar_regra(r_id, num_inputs, num_mfs_per_input, input_features_anfis)
    impacto = row['Impacto_no_Modelo']
    ativacao = row['Ativacao_Media']

    print(f"\n[Regra {r_id}] - Impacto Real: {impacto:.4f} (Ativação Média: {ativacao:.2%})")
    print(f"Lógica:{logica}")

    # Mostrar o viés (bias) da regra no consequente Sugeno
    coefs = anfis_model.consequents[r_id].detach().cpu().numpy()
    print(f"Consequente (Bias da Regra): {coefs[-1]:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- Análise 1: Distribuição de Ativação das Regras ---
plt.figure(figsize=(10, 5))
plt.stem(ranking_regras['ID_Regra'].iloc[:50], ranking_regras['Ativacao_Media'].iloc[:50])
plt.title('Top 50 Regras por Frequência de Ativação (Firing Strength)')
plt.xlabel('ID da Regra')
plt.ylabel('Ativação Média no Dataset')
plt.grid(True, alpha=0.3)
plt.show()

print("Nota: O gráfico acima mostra que o modelo baseia a maioria das decisões em um pequeno subconjunto de regras, o que é ótimo para a interpretabilidade.")

In [ ]:
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
import torch
import matplotlib.pyplot as plt

# --- Análise 2: Superfície de Decisão (Soft_F1 vs EX) ---
def plot_decision_surface(model, feat_idx1, feat_idx2, feature_names):
    x = np.linspace(0, 1, 20)
    y = np.linspace(0, 1, 20)
    X_grid, Y_grid = np.meshgrid(x, y)

    # Criar input 'base' com as médias
    base_input = X_full_tensor.mean(dim=0).clone().detach()

    z = []
    for i in range(len(x)):
        for j in range(len(y)):
            input_sample = base_input.clone()
            input_sample[feat_idx1] = torch.tensor(X_grid[i, j])
            input_sample[feat_idx2] = torch.tensor(Y_grid[i, j])

            with torch.no_grad():
                out = torch.sigmoid(model(input_sample.unsqueeze(0))).item()
            z.append(out)

    Z_grid = np.array(z).reshape(X_grid.shape)

    fig = plt.figure(figsize=(9, 7))
    ax = fig.add_subplot(111, projection='3d')
    surf = ax.plot_surface(X_grid, Y_grid, Z_grid, cmap='RdYlGn', edgecolor='none', alpha=0.8)

    ax.set_xlabel(feature_names[feat_idx1])
    ax.set_ylabel(feature_names[feat_idx2])
    ax.set_zlabel('Probabilidade de Maioria')
    ax.set_title(f'Superfície de Decisão: {feature_names[feat_idx1]} vs {feature_names[feat_idx2]}')
    fig.colorbar(surf, ax=ax, shrink=0.5, aspect=20)

    # --- NOVO: Salvar como PDF ---
    nome_arquivo = f"results/Superficie_Decisao_{feature_names[feat_idx1]}_vs_{feature_names[feat_idx2]}.pdf"
    plt.savefig(nome_arquivo, format='pdf', bbox_inches='tight')

    plt.show()

# Identificar índices de Soft_F1 e EX
idx_soft = input_features_anfis.index('Soft_F1')
idx_ex = input_features_anfis.index('EX')

plot_decision_surface(anfis_model, idx_soft, idx_ex, input_features_anfis)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

def plot_multiple_surfaces_c4(model, base_feat_name, other_features, input_features, fixed_values={'EX': 0.0}):
    """
    Gera superfícies de decisão 3D para base_feat_name vs cada métrica em other_features.
    Mantém métricas fixas (como EX=0).
    """
    model.eval()
    num_plots = len(other_features)
    base_idx = input_features.index(base_feat_name)

    # Criar um input base (usando médias do dataset original)
    base_input_template = X_full_tensor.mean(dim=0).clone().detach()

    # Aplicar valores fixos (ex: EX = 0)
    for feat, val in fixed_values.items():
        if feat in input_features:
            base_input_template[input_features.index(feat)] = val

    x_range = np.linspace(0, 1, 20)
    y_range = np.linspace(0, 1, 20)
    X_grid, Y_grid = np.meshgrid(x_range, y_range)

    for other_feat in other_features:
        if other_feat == base_feat_name:
            continue

        other_idx = input_features.index(other_feat)
        z = []

        for i in range(len(x_range)):
            for j in range(len(y_range)):
                input_sample = base_input_template.clone()
                input_sample[base_idx] = torch.tensor(X_grid[i, j])
                input_sample[other_idx] = torch.tensor(Y_grid[i, j])

                with torch.no_grad():
                    out = torch.sigmoid(model(input_sample.unsqueeze(0))).item()
                z.append(out)

        Z_grid = np.array(z).reshape(X_grid.shape)

        fig = plt.figure(figsize=(9, 7))
        ax = fig.add_subplot(111, projection='3d')
        surf = ax.plot_surface(X_grid, Y_grid, Z_grid, cmap='RdYlGn', edgecolor='none', alpha=0.8)

        ax.set_xlabel(base_feat_name)
        ax.set_ylabel(other_feat)
        ax.set_zlabel('Prob. Maioria')
        ax.set_title(f'Superfície: {base_feat_name} vs {other_feat} (Cenário EX=0)')
        fig.colorbar(surf, ax=ax, shrink=0.5, aspect=20)
        nome_arquivo = f"results/Superficie_{base_feat_name}_vs_{other_feat}.pdf"

        # Salva o gráfico como PDF ANTES do plt.show()
        # O parâmetro bbox_inches='tight' garante que as labels não fiquem cortadas
        plt.savefig(nome_arquivo, format='pdf', bbox_inches='tight')

        plt.show()

# Lista de métricas exceto Soft_F1 e EX
other_metrics = [f for f in input_features_anfis if f not in ['Soft_F1', 'EX']]

# Gerar os gráficos
plot_multiple_surfaces_c4(anfis_model, 'Stats', other_metrics, input_features_anfis, fixed_values={'EX': 0.0})

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Correlação das métricas com a saída do ANFIS
plt.figure(figsize=(10, 8))
corr_matrix = df_with_anfis_continuous_predictions[input_features_anfis + ['ANFIS_Continuous_Maioria']].corr()
sns.heatmap(corr_matrix[['ANFIS_Continuous_Maioria']].sort_values(by='ANFIS_Continuous_Maioria', ascending=False),
            annot=True, cmap='RdYlGn', vmin=-1, vmax=1)
plt.title('Correlação das Métricas')
plt.show()